In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

2026-10-16 02:39:44,523 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026-10-16 02:39:44,523 INFO sqlalchemy.engine.Engine COMMIT


✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [7]:
# 14 additional articles were added to data/external/cultpass_articles.jsonl (18 in total),
# covering refunds, blocked accounts, billing, tiers, quota, QR codes, privacy, technical issues and escalation.

In [8]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [9]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [10]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [11]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [12]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [13]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [14]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


**Ticket history and long-term memory (demo data)**

A previous, resolved ticket and a stored preference for Bob Stone, so the agents can recall them in new tickets.

In [15]:
from datetime import datetime, timedelta

bob = next(u for u in cultpass_users if u["id"] == "f556c0")

with get_session(engine) as session:
    bob_user = udahub.User(
        user_id=str(uuid.uuid4()),
        account_id=account_id,
        external_user_id=bob["id"],
        user_name=bob["name"],
    )
    past_ticket = udahub.Ticket(
        ticket_id="past-bob-login",
        account_id=account_id,
        user_id=bob_user.user_id,
        channel="email",
        created_at=datetime.now() - timedelta(days=3),
    )
    past_metadata = udahub.TicketMetadata(
        ticket_id=past_ticket.ticket_id,
        status="resolved",
        main_issue_type="login_access",
        tags="login, password",
    )
    past_messages = [
        udahub.TicketMessage(
            message_id=str(uuid.uuid4()),
            ticket_id=past_ticket.ticket_id,
            role=udahub.RoleEnum.user,
            content="I can't log in, it says my password is wrong.",
            created_at=datetime.now() - timedelta(days=3),
        ),
        udahub.TicketMessage(
            message_id=str(uuid.uuid4()),
            ticket_id=past_ticket.ticket_id,
            role=udahub.RoleEnum.ai,
            content="Try tapping 'Forgot Password' on the login screen and use your registered email. "
                    "If the email doesn't arrive, check spam or try again in a few minutes.",
            created_at=datetime.now() - timedelta(days=3, minutes=-2),
        ),
    ]
    preference = udahub.CustomerMemory(
        memory_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=bob_user.user_id,
        category="preference",
        content="Enjoys music and dance experiences, especially in Rio de Janeiro",
        source_ticket_id=past_ticket.ticket_id,
    )
    session.add_all([bob_user, past_ticket, past_metadata, *past_messages, preference])

# Tests

In [16]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [17]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='26144a5f-be3a-4d7f-8754-6481b1952ff5', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='0e479b2d-f22f-490e-b698-7eac5a988422', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='4997604a-35c3-4031-912c-721e0957179e', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='10d88fad-dbee-4587-a8ef-6be70d265636', title='How to Handle Login Issues?')>
<Knowledge(article_id='15b6befc-0595-40c2-b9e0-c2f0ffaca9e9', title='How to Cancel a Reservation')>
<Knowledge(article_id='a0d5eeab-ae99-4c36-b11d-122451a26afa', title='Refund Policy')>
<Knowledge(article_id='f4916f86-9505-4efb-8736-3cf3bf7e2ac9', title='Blocked or Suspended Accounts')>
<Knowledge(article_id='e98e9515-ca2c-458e-8b04-5c89b8c72d0c', title='Updating Payment Method and Billing Issues')>
<Knowledge(article_id='dd6317a2-7f05-4029-bdf0-c7f8af54f5fb', title='Premium Experiences and Subscription Tiers')>
<Knowledge(article_id='e185b388-c65a-4639-97fb-222

In [18]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='361d3766-8c76-45e9-a1c8-c0bd4dc2f01a', user_name='Alice Kingsley', external_user_id='a4ab87')>
<User(user_id='1621d0d7-5f79-4f59-b150-619360d0c51a', user_name='Bob Stone', external_user_id='f556c0')>


In [19]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='501c17fa-c0b9-4381-9327-80cbd60a14b4', role='user', content='I can't log in to my Cultpass ...')>


In [20]:
with get_session(engine) as session:
    for memory in session.query(udahub.CustomerMemory).all():
        print(memory)

<CustomerMemory(user_id='1621d0d7-5f79-4f59-b150-619360d0c51a', category='preference', content='Enjoys music and dance experie')>
